<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2E: *Spatial Distance Fire Data*
##### Version Number: 4.0
---
### Contents  
> *Fire Damage Spatial Join of Nearest Sampling Locations*\
> *Fire Ignition Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Fire Spread Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from the sampling grid.
### Inputs
- Main Dataset - `samples_res.csv` 
- Wildfire Damage Data - `fires_damage.csv` (Appendix B)
- Wildfire Ignition Data - `fires_ignition.csv` (Appendix B)
- Wildfire Spread Data - `fires_spread.csv` (Appendix B)
---
### Outputs  
- `samples_fires.csv` Sampling grid dataset merged with fire damage, spread, and ignition data.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

## Data Loading

In [3]:
samples = pd.read_csv("../data/processed/samples_res_distance.csv")
fire_ignition = pd.read_csv('../data/processed/fires_ignition.csv')

In [4]:
samples['Date'] = pd.to_datetime(samples['Date'])
fire_ignition['Date'] = pd.to_datetime(fire_ignition['Date'])

In [5]:
samples = samples.sort_values(
    by=["grid_id", "Date"]
)

#### Subset data for faster loops

In [6]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep].copy()

#### Set geometries
- Sample grid ArcGIS work was performed in EPSG 3310 to preserve area. 

In [7]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples['centroid_easting'], join_samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

In [8]:
fire_ignition['geometry'] = [Point(xy) for xy in zip(fire_ignition['Fire_Longitude'], fire_ignition['Fire_Latitude'])]
fire_ignition_gdf = gpd.GeoDataFrame(fire_ignition, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
fire_ignition_projected = fire_ignition_gdf.to_crs(3310)

### Main loop of Spatial Join algorithm

All three widlfire datasets use the same algorithm.
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load grid and fire centroids associated with the current date
- Create a buffer around each around each fire for current date
- Intersection spatial join of sampling centroids and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to new dataframe

## Fire Damage Spatial Join

## Distance to fires

In [9]:
def add_fire_distance_features_by_year(
    samples_gdf,
    fires_gdf,
    year,
    past_days=None
):
    # Filter by year
    samples_year = samples_gdf[samples_gdf['Date'].dt.year == year]
    fires_year = fires_gdf[fires_gdf['Date'].dt.year == year]

    # if no samples this year skip
    if samples_year.empty or fires_year.empty:
        return samples_gdf

    for dt in samples_year['Date'].unique():
        
        # Subset grid centroids for date
        samples_today = samples_year[samples_year['Date'] == dt]
        
        # Subset fires on this date
        fires_today = fires_year[fires_year['Date'] == dt]

        if samples_today.empty:
            continue

        # Find distances to fires on the same day
        if not fires_today.empty:
            dist_matrix = np.array([
                fires_today.geometry.distance(s) for s in samples_today.geometry
            ])
            
            # assign distance to closest fire to dataframe
            samples_gdf.loc[samples_gdf['Date'] == dt, 'dist_to_fires_same_day'] = dist_matrix.min(axis=1)
            
            # assign the average distance to all fires to dataframe
            samples_gdf.loc[samples_gdf['Date'] == dt, 'avg_dist_to_fires_same_day'] = dist_matrix.mean(axis=1)
        else: # no data
            samples_gdf.loc[samples_gdf['Date'] == dt, 'dist_to_fires_same_day'] = np.nan
            samples_gdf.loc[samples_gdf['Date'] == dt, 'avg_dist_to_fires_same_day'] = np.nan

        # Subset past fires this year
        past_fires = fires_gdf[fires_gdf['Date'] < dt]
        
        if past_days is not None:
            start_date = dt - pd.Timedelta(days=past_days)
            past_fires = past_fires[past_fires['Date'] >= start_date]

        if past_fires.empty:
            samples_gdf.loc[samples_gdf['Date'] == dt, 'avg_dist_to_past_fires'] = np.nan
        else:
            dist_matrix_past = np.array([
                past_fires.geometry.distance(s) for s in samples_today.geometry
            ])
            samples_gdf.loc[samples_gdf['Date'] == dt, 'avg_dist_to_past_fires'] = dist_matrix_past.mean(axis=1)

    return samples_gdf


In [10]:
premerged = samples_gdf.copy()  # Snapshot for post merge checks

years = sorted(samples_gdf['Date'].dt.year.unique())

for yr in years:
    samples_gdf = add_fire_distance_features_by_year(
        samples_gdf,
        fire_ignition_projected,
        year=yr,
        past_days=30  # optional rolling window
    )

In [11]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  12508


**Note:** NA values represent days without fires, assign the distance to 1,000,000 meters

In [12]:
samples_gdf = samples_gdf.fillna(1_000_000 )

## Merge back to main dataset

In [13]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

In [14]:
# Merge on BOTH station and date
samples_fires = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [15]:
post_merge_check(samples_fires, samples)

Premerged shape:  (608880, 77)
Merged shape:  (608880, 80)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


## Calculate days since last fire

In [16]:
def compute_days_since_last_fire_from_counts(samples_gdf):

    # Sort by grid and date
    samples_sorted = samples_gdf.sort_values(['grid_id', 'Date']).copy()

    # Initialize output
    days_since_last_fire = []

    # Group by grid
    for grid, group in samples_sorted.groupby('grid_id'):
        
        last_fire_day = None
        

        for date, fire_count in zip(group['Date'], group['fire_count']):
            if last_fire_day is None:
                days_since_last_fire.append(np.nan)
            else:
                days_since_last_fire.append((date - last_fire_day).days)

            # Update last_fire_day if there was a fire today
            if fire_count > 0:
                last_fire_day = date

    return pd.Series(days_since_last_fire, index=samples_sorted.index)


In [17]:
samples_fires['days_since_last_fire'] = compute_days_since_last_fire_from_counts(samples_fires)

In [18]:
samples_fires = samples_fires.fillna(9999)

In [19]:
samples_fires.isna().sum()

Sample_ID                              0
Date                                   0
Burning Index                          0
Energy Release Component               0
Actual Evapotranspiration              0
                                      ..
avg_dist_to_all_reservoirs_same_day    0
dist_to_fires_same_day                 0
avg_dist_to_fires_same_day             0
avg_dist_to_past_fires                 0
days_since_last_fire                   0
Length: 81, dtype: int64

## Export File

In [20]:
samples_fires.to_csv('../data/processed/samples_fires_distance.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
